# Kandinsky Game→Audio — Step 8: Pass Game Tokens Through PiFlow

Steps 3–7 validated the Game→Audio branch and post-load adapter lifecycle at block and tiny-DiT level. This step connects the structured event tokens to the pinned Kandinsky PiFlow sampler's actual model call.

The notebook:
- patches only scratch copies in Colab;
- forwards `game_tokens` and `game_attn_mask` on every PiFlow denoising step;
- executes two short CPU-sized sampler steps on a tiny random DiT and latent bundle.

**No pretrained weights, training, production pipeline edits, or WAV generation are used.** The latent outputs are not listenable audio.

## What a pass means

- The PiFlow sampler invokes the DiT with the same game-token tensor and mask at each step.
- Without event tokens, the sampler still runs.
- With a manually opened demo gate, the final audio latent changes.

This proves only API plumbing and a numerical effect in a tiny random model. It does not show that the event representation is meaningful or that decoded audio sounds good.

## 1. Environment

Use a fresh or restarted CPU runtime. The source import uses an isolated Kandinsky package namespace.

In [ ]:
import importlib.util
import subprocess
import sys
import types
from pathlib import Path

import torch
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device: CPU is sufficient for this small forward-path check.")

## 2. Fetch the pinned source

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step8-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True,
    )

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
source = DIT_PATH.read_text(encoding="utf-8")
assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT
print("Pinned source:", K6_COMMIT)
print("Original source:", DIT_PATH)

## 3. Add Game→Audio to a scratch DiT source copy

This is the same block and forward-path patch used in Step 4. The official source checkout remains untouched.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

# Add the optional event-token width to the real fused block.
patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "fused-block constructor",
)

# Create the Game→Audio attention branch only when game_dim is configured.
patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "fused-block Game→Audio modules",
)

# Extend the fused block call. These are also used for audio-only calls.
patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "    ):\n",
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ):\n",
    "fused-block forward signature",
)

# Apply the branch after existing text/AV updates and before the audio FFN.
patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_tokens = game_tokens.to(\n"
    "                device=aud.device, dtype=self.game_to_audio.to_key.weight.dtype\n"
    "            )\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = game_attn_mask.to(device=aud.device, dtype=torch.bool)\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "fused-block Game→Audio insertion",
)

# Add game_dim to DiffusionTransformer3D and pass it to every fused block.
patched = replace_once(
    patched,
    "        visual_token_type_num_embeddings: int = 0,\n"
    "    ):\n",
    "        visual_token_type_num_embeddings: int = 0,\n"
    "        game_dim: int | None = None,\n"
    "    ):\n",
    "DiT constructor signature",
)
patched = replace_once(
    patched,
    "        self.is_multimodal  = is_multimodal\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "        self.is_multimodal  = is_multimodal\n"
    "        self.game_dim       = game_dim\n"
    "        if game_dim is not None and not is_multimodal:\n"
    "            raise ValueError(\"game_dim is supported only by the multimodal audio model.\")\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "DiT game_dim configuration",
)
patched = replace_once(
    patched,
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n",
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n"
    "                    game_dim=game_dim,\n",
    "pass game_dim to fused-block stack",
)

# Thread game tokens through the audio-only fused-block loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "single-modality block-loop signature",
)
patched = replace_once(
    patched,
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                    )\n",
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                        game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "                    )\n",
    "audio-only fused-block call",
)

# Thread the same tensors through the joint video+audio loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "fused block-loop signature",
)
patched = replace_once(
    patched,
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "joint fused-block call",
)

# Add public forward arguments and validate their basic contract.
patched = replace_once(
    patched,
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "public forward signature",
)
patched = replace_once(
    patched,
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens.\")\n"
    "        if game_tokens is not None:\n"
    "            if not self.is_multimodal or self.game_dim is None:\n"
    "                raise ValueError(\"Configure game_dim on a multimodal model before passing game_tokens.\")\n"
    "            if x_audio is None:\n"
    "                raise ValueError(\"Game conditioning requires x_audio.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim].\")\n"
    "            audio_batch = x_audio.shape[0] if x_audio.ndim == 3 else 1\n"
    "            if game_tokens.shape[0] != audio_batch:\n"
    "                raise ValueError(\"game_tokens and x_audio must have the same batch size.\")\n"
    "            if game_attn_mask is not None:\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means a valid event token.\")\n"
    "                if game_attn_mask.shape[-1] != game_tokens.shape[1]:\n"
    "                    raise ValueError(\"The final game_attn_mask dimension must match the event count.\")\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "forward argument validation",
)

# Pass the optional condition to audio-only and joint forward paths.
patched = replace_once(
    patched,
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "audio-only DiT call",
)
patched = replace_once(
    patched,
    "            vis_rope, aud_rope, attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "            vis_rope, aud_rope, attn_mask,\n"
    "            game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "joint DiT call",
)

PATCH_PATH = DIT_PATH.with_name("dit_game_full.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Patched scratch copy:", PATCH_PATH)
print("Original source unchanged:", DIT_PATH.read_text(encoding="utf-8") == source)

## 4. Patch the PiFlow sampler's condition interface

The pinned sampler already passes text, video/audio latents, time, RoPE, and visual token IDs to the DiT. We add two optional arguments and forward them with the existing model call.

In [ ]:
PIFLOW_PATH = K6_DIR / "kandinsky/core/algo/piflow_sampler.py"
piflow_source = PIFLOW_PATH.read_text(encoding="utf-8")

piflow_patched = replace_once(
    piflow_source,
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    game_tokens: torch.Tensor | None = None,\n"
    "    game_attn_mask: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "PiFlow optional game condition arguments",
)

piflow_patched = replace_once(
    piflow_patched,
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "            game_tokens=game_tokens,\n"
    "            game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "forward game condition to the DiT on every PiFlow step",
)

PIFLOW_GAME_PATH = PIFLOW_PATH.with_name("piflow_sampler_game.py")
PIFLOW_GAME_PATH.write_text(piflow_patched, encoding="utf-8")
print("Scratch PiFlow sampler:", PIFLOW_GAME_PATH)
print("Official sampler unchanged:", PIFLOW_PATH.read_text(encoding="utf-8") == piflow_source)

## 5. Import the actual DiT and PiFlow source

In [ ]:
if "kandinsky" in sys.modules:
    raise RuntimeError(
        "A Kandinsky package is already imported. Restart the runtime, then run all cells from the top."
    )

def register_package(name, path):
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    module.__package__ = name
    sys.modules[name] = module

register_package("kandinsky", K6_DIR / "kandinsky")
register_package("kandinsky.core", K6_DIR / "kandinsky/core")
register_package("kandinsky.core.components", K6_DIR / "kandinsky/core/components")
register_package("kandinsky.core.components.attention", K6_DIR / "kandinsky/core/components/attention")
register_package("kandinsky.runtime", K6_DIR / "kandinsky/runtime")
register_package("kandinsky.runtime.kernels", K6_DIR / "kandinsky/runtime/kernels")

def pinned_sdpa(q, k, v, attn_mask=None):
    # Kandinsky attention tensors are [batch, sequence, heads, head_dim].
    q = q.transpose(1, 2).contiguous()
    k = k.transpose(1, 2).contiguous()
    v = v.transpose(1, 2).contiguous()
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask).transpose(1, 2).contiguous()

class SDPAOnlyEngine:
    def __init__(self, engine="auto"):
        if engine not in {"auto", "sdpa"}:
            raise ValueError("This notebook supports only the SDPA attention path.")
    def get_attention(self):
        return pinned_sdpa

engine_module = types.ModuleType("kandinsky.runtime.kernels.attention_engine")
engine_module._sdpa = pinned_sdpa
engine_module.SelfAttentionEngine = SDPAOnlyEngine
engine_module.resolve_attention_engine = lambda requested: "sdpa"
sys.modules[engine_module.__name__] = engine_module

def load_dit_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

base_dit = load_dit_module(
    "kandinsky.core.components.dit_base", DIT_PATH
)
game_dit = load_dit_module(
    "kandinsky.core.components.dit_game_full", PATCH_PATH
)
print("Baseline model class:", base_dit.DiffusionTransformer3D)
print("Patched model class:", game_dit.DiffusionTransformer3D)

# Load the patched PiFlow sampler under Kandinsky's source package namespace.
piflow_game = load_dit_module(
    "kandinsky.core.algo.piflow_sampler_game", PIFLOW_GAME_PATH
)
from kandinsky.core.types import LatentBundle
print("Loaded PiFlow function:", piflow_game.piflow_denoise_loop.__name__)

## 6. Build a tiny game-enabled DiT

In [ ]:
torch.manual_seed(808)

BASE_CONFIG = dict(
    in_visual_dim=4,
    out_visual_dim=4,
    in_text_dim=12,
    in_text_dim2=6,
    time_dim=16,
    patch_size=(1, 2, 2),
    model_dim=24,
    ff_dim=48,
    num_text_blocks=1,
    num_visual_blocks=2,
    axes_dims=(2, 2, 2),
    visual_cond=False,
    is_multimodal=True,
    in_audio_dim=5,
    out_audio_dim=5,
    model_dim_a=24,
    time_dim_a=16,
    ff_dim_a=48,
    axes_dims_a=(2, 2, 2),
    attention_engine="sdpa",
)
GAME_DIM = 7
model = game_dit.DiffusionTransformer3D(
    **BASE_CONFIG,
    game_dim=GAME_DIM,
).eval()

# Open only the last branch gate for a visible, untrained plumbing demonstration.
with torch.no_grad():
    model.visual_transformer_blocks[-1].game_audio_modulation.out_layer.bias.fill_(0.1)

print("Tiny model blocks:", len(model.visual_transformer_blocks))
print("No pretrained weights loaded.")

## 7. Prepare a miniature video/audio latent bundle

PiFlow requires both video and audio latents plus cumulative sequence offsets. These random tensors are synthetic hidden-space inputs, not pixels or playable audio.

In [ ]:
BATCH = 1
VIDEO_FRAMES = 2
AUDIO_LEN = 8
TEXT_LEN = 4
GAME_LEN = 3

video = torch.randn(BATCH, VIDEO_FRAMES, 2, 2, BASE_CONFIG["in_visual_dim"])
audio = torch.randn(BATCH, AUDIO_LEN, BASE_CONFIG["in_audio_dim"])
video_cu = torch.tensor([0, VIDEO_FRAMES], dtype=torch.int32)
audio_cu = torch.tensor([0, AUDIO_LEN], dtype=torch.int32)

text_embeds = {
    "text_embeds": torch.randn(BATCH, TEXT_LEN, BASE_CONFIG["in_text_dim"]),
    "pooled_embed": torch.randn(BATCH, BASE_CONFIG["in_text_dim2"]),
}
game_tokens = torch.randn(BATCH, GAME_LEN, GAME_DIM)
game_valid = torch.tensor([[True, True, False]])
game_attn_mask = game_valid[:, None, None, :]

video_positions = [torch.arange(VIDEO_FRAMES), torch.arange(1), torch.arange(1)]
visual_rope = model.visual_rope((VIDEO_FRAMES, 1, 1), video_positions)
audio_rope = model.audio_rope(torch.arange(AUDIO_LEN))
positions = torch.arange(TEXT_LEN)
text_rope = [
    model.video_text_rope(positions),
    model.audio_text_rope(positions),
]

def new_bundle():
    return LatentBundle(
        video=video.clone(),
        audio=audio.clone(),
        video_cu_seqlens=video_cu.clone(),
        audio_cu_seqlens=audio_cu.clone(),
    )

print("Video latent:", tuple(video.shape))
print("Audio latent:", tuple(audio.shape))
print("Game token + mask:", tuple(game_tokens.shape), tuple(game_attn_mask.shape))

## 8. Record exactly what the sampler forwards

In [ ]:
class RecordingDiT(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner
        self.condition_calls = []

    @property
    def is_multimodal(self):
        return self.inner.is_multimodal

    @property
    def visual_cond(self):
        return self.inner.visual_cond

    def forward(self, **kwargs):
        tokens = kwargs.get("game_tokens")
        mask = kwargs.get("game_attn_mask")
        self.condition_calls.append({
            "tokens": None if tokens is None else tuple(tokens.shape),
            "mask": None if mask is None else tuple(mask.shape),
        })
        return self.inner(**kwargs)

def run_piflow(*, with_game_condition):
    recorder = RecordingDiT(model)
    result = piflow_game.piflow_denoise_loop(
        bundle=new_bundle(),
        dit=recorder,
        text_embeds=text_embeds,
        visual_rope=visual_rope,
        audio_rope=audio_rope,
        text_rope=text_rope,
        num_steps=2,
        scheduler_scale=1.0,
        sample_video=True,
        sample_audio=True,
        game_tokens=game_tokens if with_game_condition else None,
        game_attn_mask=game_attn_mask if with_game_condition else None,
        num_policy_substeps=1,
    )
    return result, recorder.condition_calls

without_game, calls_without = run_piflow(with_game_condition=False)
with_game, calls_with = run_piflow(with_game_condition=True)

assert len(calls_without) == 2
assert all(call["tokens"] is None for call in calls_without)
assert len(calls_with) == 2
assert all(call["tokens"] == (BATCH, GAME_LEN, GAME_DIM) for call in calls_with)
assert all(call["mask"] == (BATCH, 1, 1, GAME_LEN) for call in calls_with)

audio_latent_delta = (with_game.audio - without_game.audio).abs().mean().item()
assert audio_latent_delta > 0
assert torch.isfinite(with_game.audio).all()
assert torch.isfinite(with_game.video).all()

print("Calls without game condition:", calls_without)
print("Calls with game condition:", calls_with)
print("Output video latent:", tuple(with_game.video.shape))
print("Output audio latent:", tuple(with_game.audio.shape))
print("Mean audio-latent change with demo gate:", audio_latent_delta)
print("PiFlow condition forwarding and latent output: passed")

## What Step 8 demonstrates — and what it does not

**Demonstrated if all cells pass:** on every PiFlow step, the sampler forwards the same event tokens and boolean mask to the model; the tiny DiT accepts them; and the manually opened branch changes the audio latent.

**Not demonstrated:** full Kandinsky pipeline API integration, real video/audio VAE latents, pretrained checkpoint loading, trained Game→Audio weights, or listenable audio quality. The model and latent inputs are random and tiny.

**Next:** only after this sampler seam passes should the game condition be exposed in a scratch pipeline call. Then test with a real permitted checkpoint and decode/listen; training and game-event data design remain separate work.